# 重叠收益的上界

> 对应博客篇目：《[性能剖析（01）：训练栈 profiling](https://lrypcy.github.io/2026/09/29/profiling-01-training/)》
>
> 代码逐字摘自博客仓库 `tools/profiling_bench/overlap_bound.py` 与 `tools/megatron_bench/comm.py`（[lrypcy/ipynbs](https://github.com/lrypcy/ipynbs)）。纯 CPU，无第三方依赖。

## 这个 notebook 回答三个问题

1. **一次集合通信里，有多少是「可以藏」的？**
   $\alpha$-$\beta$ 模型把时间分成两截：`steps·α`（延迟项，依赖链，与计算量无关）与 `bytes/BW`（带宽项，有独立计算可跑就能藏住）。两项相等发生在 $D^* = n \cdot \alpha \cdot BW$。**这条比「通信占比」有用得多**——消息小于 $D^*$ 时是延迟主导，做重叠收益很小。

2. **一次迭代里通信的理论下界是什么？**
   完全不重叠 $T = C + M$；完美重叠 $T \ge \max(C, M)$；现实 $T \approx C + M - \eta\min(C, M)$。

3. **nccl-tests 的 algbw 和 busbw 差多少、差在哪。**
   $\text{busbw} = \text{algbw} \times 2(N-1)/N$（AllReduce）。**但 busbw 在 NVLS / CollNet / 分层算法下不再对应单一物理链路**，拿它反推线速会把「算法利用率」读成「链路带宽」。

## 第 0 步 · 环境与共享记账

**这步在做什么**：把两个依赖摊平——

- `configs`：模型与硬件标称规格（Llama-3-8B / A100 / H100 / B200）
- `comm`：集合通信的 $\alpha$-$\beta$ 代价模型。约定**带宽用单向链路带宽**，时间公式采用 nccl-tests 的 bus-bandwidth 口径

集合通信的单卡搬运量：

$$\text{bytes} = \begin{cases} \frac{2(n-1)}{n} D & \text{AllReduce} \\ \frac{n-1}{n} D & \text{ReduceScatter / AllGather / All-to-All}\end{cases} \qquad t = \text{steps} \cdot \alpha + \frac{\text{bytes}}{BW}$$

In [1]:
from dataclasses import dataclass, field
from typing import Dict, Optional
from dataclasses import dataclass
from typing import Dict, List
import argparse
import os
import sys

In [2]:
# ---- 摘自 tools/megatron_bench/configs.py ----
@dataclass
class ModelSpec:
    """一个 dense Transformer（GPT 系）的规格。"""

    name: str
    hidden: int          # h
    n_layers: int        # L
    n_heads: int         # attention heads
    ffn_hidden: int      # I, SwiGLU 的中间宽度（已经是乘过 ffn_mult 的值）
    vocab: int
    seq_len: int
    n_kv_heads: Optional[int] = None   # None 表示 MHA；否则是 GQA 的 kv 组数
    gated_mlp: bool = True             # True=SwiGLU(gate+up+down 三个矩阵)；False=GELU(up+down 两个)
    source: str = ""

    def __post_init__(self):
        if self.n_kv_heads is None:
            self.n_kv_heads = self.n_heads
        assert self.hidden % self.n_heads == 0, "hidden 必须能被 head 数整除"

    @property
    def head_dim(self) -> int:
        return self.hidden // self.n_heads

    @property
    def kv_dim(self) -> int:
        return self.n_kv_heads * self.head_dim

    def n_params(self, untie_embeddings: bool = False) -> int:
        """非 embedding 参数量 + embedding。用于和官方公布的参数量对账。"""
        attention = self.hidden * self.hidden          # W_q
        attention += 2 * self.hidden * self.kv_dim     # W_k, W_v
        attention += self.hidden * self.hidden         # W_o
        if self.gated_mlp:
            mlp = 2 * self.hidden * self.ffn_hidden    # gate + up
            mlp += self.ffn_hidden * self.hidden       # down
        else:
            mlp = 2 * self.hidden * self.ffn_hidden    # up + down
        per_layer = attention + mlp
        embed = self.vocab * self.hidden
        total = self.n_layers * per_layer + embed
        if not untie_embeddings:
            total += embed                             # output layer 独立
        return total

    def n_params_no_embedding(self) -> int:
        attention = 2 * self.hidden * self.hidden + 2 * self.hidden * self.kv_dim
        mlp = self.hidden * self.ffn_hidden * (3 if self.gated_mlp else 2)
        return self.n_layers * (attention + mlp)



MODELS: Dict[str, ModelSpec] = {
    # Llama-2 7B 结构：hidden 4096 / 32 层 / MHA / SwiGLU 中间 11008
    "llama-7b": ModelSpec(
        name="llama-7b", hidden=4096, n_layers=32, n_heads=32,
        ffn_hidden=11008, vocab=32000, seq_len=4096,
        source="Llama-2-7B config (HuggingFace meta-llama/Llama-2-7b-hf)",
    ),
    # Llama-3 8B 系：改了 GQA（8 个 kv head）+ 更大词表
    "llama3-8b": ModelSpec(
        name="llama3-8b", hidden=4096, n_layers=32, n_heads=32, n_kv_heads=8,
        ffn_hidden=14336, vocab=128256, seq_len=8192,
        source="Llama-3-8B config (meta-llama/Meta-Llama-3-8B)",
    ),
    # Qwen2.5-14B 系 calcifications（含 GQA）
    "qwen-14b": ModelSpec(
        name="qwen-14b", hidden=5120, n_layers=48, n_heads=40, n_kv_heads=8,
        ffn_hidden=13824, vocab=152064, seq_len=8192,
        source="Qwen2.5-14B config (Qwen/Qwen2.5-14B)",
    ),
    # GPT-3 175B：Megatron-LM examples/pretrain_gpt3_175B.sh
    "gpt3-175b": ModelSpec(
        name="gpt3-175b", hidden=12288, n_layers=96, n_heads=96,
        ffn_hidden=49152, vocab=50257, seq_len=2048, gated_mlp=False,
        source="Megatron-LM examples/gpt3/train_gpt3_175b_distributed.sh",
    ),
}



@dataclass
class HardwareSpec:
    """一张卡的标称 specs。带宽统一用单向 GB/s，避免 bidir 口径混淆。"""

    name: str
    bf16_tflops: float      # dense BF16 算力峰值 (TFLOP/s)
    hbm_gbs: float          # HBM 带宽 (GB/s)
    mem_gb: float           # 单卡显存 (GiB)
    nvlink_gbs: float       # NVLink 单向聚合带宽 (GB/s)
    ib_gbs: float           # 节点间网络单向带宽 (GB/s)
    gpus_per_node: int
    source: str = ""



HARDWARE: Dict[str, HardwareSpec] = {
    "a100-80g": HardwareSpec(
        name="a100-80g", bf16_tflops=312.0, hbm_gbs=2039.0, mem_gb=80.0,
        nvlink_gbs=300.0, ib_gbs=25.0, gpus_per_node=8,
        source="NVIDIA A100 datasheet (BF16 dense 312 TFLOP/s, HBM2e 2.0 TB/s)",
    ),
    "h100-80g": HardwareSpec(
        name="h100-80g", bf16_tflops=989.0, hbm_gbs=3350.0, mem_gb=80.0,
        nvlink_gbs=450.0, ib_gbs=50.0, gpus_per_node=8,
        source="NVIDIA H100 SXM datasheet (BF16 dense 989 TFLOP/s, HBM3 3.35 TB/s)",
    ),
    "b200-180g": HardwareSpec(
        name="b200-180g", bf16_tflops=2250.0, hbm_gbs=8000.0, mem_gb=180.0,
        nvlink_gbs=900.0, ib_gbs=100.0, gpus_per_node=8,
        source="NVIDIA B200 datasheet (BF16 dense 2.25 PFLOP/s, HBM3e 8 TB/s)",
    ),
}



@dataclass
class BenchConfig:
    """一次求解的完整输入。"""

    model: str = "llama-7b"
    hardware: str = "h100-80g"
    tp: int = 1
    cp: int = 1
    pp: int = 1
    ep: int = 1
    dp: int = 1
    num_nodes: int = 1
    micro_batch: int = 4
    grad_accum: int = 8
    seq_len: Optional[int] = None
    recompute: str = "none"          # none | full | selective
    use_distributed_optimizer: bool = True
    bytes_per_param_elem: int = 2    # bf16 = 2, fp32 = 4
    extra_notes: list = field(default_factory=list)

    def resolved_seq_len(self) -> int:
        return self.seq_len or MODELS[self.model].seq_len

    @property
    def world_size(self) -> int:
        """总卡数 = 各并行维乘积。dp 一般是被最后算出来的那一维。"""
        return self.tp * self.cp * self.pp * self.ep * self.dp

In [3]:
# ---- 摘自 tools/megatron_bench/comm.py ----
def collective_bytes(kind: str, d: float, n: int) -> float:
    """单卡链路上的数据搬运字节数。d 是集合要处理的总数据量。"""
    if n <= 1:
        return 0.0
    kinds = {
        "all_reduce": 2.0 * (n - 1) / n * d,
        "reduce_scatter": (n - 1) / n * d,
        "all_gather": (n - 1) / n * d,
        "all_to_all": (n - 1) / n * d,
        "reduce": (n - 1) / n * d,
        "broadcast": d,
    }
    if kind not in kinds:
        raise ValueError("unknown collective: %s" % kind)
    return kinds[kind]



def collective_time_s(kind: str, d: float, n: int, bw_gbs: float,
                      alpha_us: float = 2.0) -> float:
    """一次集合通信的估计时长。n<=1 直接返回 0。"""
    if n <= 1:
        return 0.0
    steps = {
        "all_reduce": 2 * (n - 1),
        "reduce_scatter": n - 1,
        "all_gather": n - 1,
        "all_to_all": n - 1,
        "reduce": n - 1,
        "broadcast": n - 1,
    }[kind]
    return steps * alpha_us * 1e-6 + collective_bytes(kind, d, n) / (bw_gbs * 1e9)



@dataclass
class CommEvent:
    group: str        # tp / pp / dp / ep
    kind: str
    size_bytes: float
    count_per_layer: int
    note: str = ""

    def total_bytes(self, n: int) -> float:
        return collective_bytes(self.kind, self.size_bytes, n) * self.count_per_layer



def tensor_parallel_events(cfg: BenchConfig, sequence_parallel: bool,
                           bytes_elem: int = 2) -> List[CommEvent]:
    """一个 transformer 层的 TP 通信事件清单。

    TP only（无 SP）:  每层 4 次 All-Reduce
        —— QKV / MLP-up (ColumnParallel) 的 backward g 各一次（共 2 次）
        —— attn-dense / MLP-down (RowParallel) 的 forward f 各一次（共 2 次）
        每次搬满一个 [s, b, h] 激活：2(n-1)/n · d。

    TP + SP: 同一个 f / g 算子改变形态，**总字节数不变**。
        - f（TP 区入口）：前向 identity → all-gather；反向 identity → reduce-scatter
        - g（TP 区出口）：前向 all-reduce → reduce-scatter；反向 identity → all-gather
        于是每层：4 次 AG + 4 次 RS，每次 (n-1)/n · d
                  合计 8·(n-1)/n·d
        而 baseline TP 的 4 次 AR = 4·2(n-1)/n·d = 8·(n-1)/n·d
        —— **完全相同**。理由就是恒等式 AR(d) = RS(d) + AG(d)。

    2026-09-29 校准（此前标注"待校准"，现已核验）：
      * 恒等式 AR = RS ∘ AG 决定总量守恒；SP 分的是**激活显存**（LN/dropout/
        residual 从 O(sbh)  replicated 变成 (s/t, b, h)，缩小 t 倍），不是通信量。
      * Korthikanti et al. 的 SP 描述见 arXiv:2205.05198；对其通信条数的独立
        复述见 arXiv:2311.02382 §3（"8 global communications per attention
        layer：4 in forward pass and 4 in backward pass"），与本函数一致。
      * 因此**不要指望开 SP 降低通信量**。SP 的实际收益是显存 + 允许关掉
        activation recompute（recompute 减少才是它提速的来源）。
    """
    m = MODELS[cfg.model]
    h = m.hidden
    T = cfg.micro_batch * (cfg.resolved_seq_len() // cfg.cp)
    d = T * h * bytes_elem

    if not sequence_parallel:
        return [
            CommEvent("tp", "all_reduce", d, 2, "两层 ColumnParallel 的 backward g"),
            CommEvent("tp", "all_reduce", d, 2, "两层 RowParallel 的 forward f"),
        ]

    if cfg.tp <= 1:
        return []
    return [
        CommEvent("tp", "all_gather", d, 4,
                  "f 前向 AG + g 反向 AG（attn 与 MLP 各一对）"),
        CommEvent("tp", "reduce_scatter", d, 4,
                  "g 前向 RS + f 反向 RS（attn 与 MLP 各一对）"),
    ]

In [4]:
# ---- 模块名垫片：让原脚本里的 `megatron_bench.comm.X` / `configs.X` 原样可用 ----
import types

configs = types.SimpleNamespace(BenchConfig=BenchConfig, MODELS=MODELS,
                                HARDWARE=HARDWARE)
comm = types.SimpleNamespace(collective_bytes=collective_bytes,
                             collective_time_s=collective_time_s,
                             tensor_parallel_events=tensor_parallel_events)

In [5]:
# ---- 脚本 CLI 的默认参数（与 overlap_bound.py 的 argparse 一致）----
ARGS = argparse.Namespace(
    alpha_us=2.0, model="llama3-8b", hardware="h100-80g",
    tp=8, cp=1, pp=1, dp=8, micro_batch=4, grad_accum=8,
    seq_len=None, mfu=0.40, dump=None)

## 第 1 节 · $\alpha$ 项与 $\beta$ 项的交叉点 $D^* = n \cdot \alpha \cdot BW$

**这步在做什么**：以 AllReduce ring 为例，

$$t = 2(n-1)\alpha + \frac{2(n-1)}{n}\frac{D}{BW} \qquad \text{两项相等} \Rightarrow \alpha = \frac{D}{n \cdot BW} \Rightarrow D^* = n \cdot \alpha \cdot BW$$

AllGather / ReduceScatter 的 steps 是 $(n-1)$，比值同为 $n$，结果一样。

读法：

- $D^*$ **以下 = 延迟主导**。此时改重叠、加 buffer 都没多少用，真正有效的是「合并小消息」或换算法（NVLS）。
- $D^*$ **以上 = 带宽主导**。这才是重叠能发挥作用的区间。
- $D^*$ 与 $n$ 成正比：卡越多，依赖链越长，小消息越吃亏。

In [6]:
# ---- 摘自 tools/profiling_bench/overlap_bound.py ----
def crossover_bytes(n, alpha_us, bw_gbs):
    """α 项（依赖链延迟）与 β 项（带宽搬运）相等的消息量。

    推导（以 AllReduce ring 为例）：
        t = 2(n-1)α + 2(n-1)/n · D/BW
        两项相等  ⇒  α = D/(n·BW)  ⇒  D* = n·α·BW
    AllGather / ReduceScatter 的 steps 是 (n-1)，比值同为 n，结果一样。
    """
    return n * alpha_us * 1e-6 * bw_gbs * 1e9



def section_crossover(alpha_us=2.0):
    print("=" * 78)
    print("第 1 节  α 项与 β 项的交叉点：D* = n · α · BW")
    print("=" * 78)
    print("α = %.1f us（每次 ring 步的固定延迟，取 nccl-tests 常见量级）\n"
          % alpha_us)
    print("%-12s %-10s %8s %14s %14s" %
          ("硬件", "链路", "n", "BW (GB/s)", "D* (bytes)"))
    rows = []
    for hw_name, hw in HARDWARE.items():
        for link, bw in (("NVLink", hw.nvlink_gbs), ("IB", hw.ib_gbs)):
            for n in (8, 64):
                d = crossover_bytes(n, alpha_us, bw)
                rows.append((hw_name, link, n, bw, d))
                print("%-12s %-10s %8d %14.0f %14.0f"
                      % (hw_name, link, n, bw, d))
    print("""
读法：
  * D* 以下 = 延迟主导。此时改重叠、加 buffer 都没多少用，
    真正有效的是"合并小消息"（把 N 个小 AllReduce 合成一个）或换算法（NVLS）。
  * D* 以上 = 带宽主导。这才是重叠能发挥作用的区间。
  * D* 与 n 成正比：卡越多，依赖链越长，小消息越吃亏。
  * IB 的 D* 只有几百 KB（25 GB/s 档约 0.2 MB），
    而 NVLink 的 D* 高出两个数量级——所以"跨机通信藏不住"的第一个原因是
    它本来就落在延迟主导区，第二个原因才是带宽窄。
  * 数据并行梯度 reduce 的每层消息 = 该层参数字节数（bf16 下 7B 模型的
    attention 层约 25 MB / MLP 层约 88 MB），远在 D* 之上 → 带宽主导，可重叠。
    TP 每层每次 AR 的消息量小得多，正好落在 D* 附近 → 这就是
    `tp_comm_overlap` 收益不稳定、且强依赖 shape 的原因。""")
    return rows



section_crossover(ARGS.alpha_us)

第 1 节  α 项与 β 项的交叉点：D* = n · α · BW
α = 2.0 us（每次 ring 步的固定延迟，取 nccl-tests 常见量级）

硬件           链路                n      BW (GB/s)     D* (bytes)
a100-80g     NVLink            8            300        4800000
a100-80g     NVLink           64            300       38400000
a100-80g     IB                8             25         400000
a100-80g     IB               64             25        3200000
h100-80g     NVLink            8            450        7200000
h100-80g     NVLink           64            450       57600000
h100-80g     IB                8             50         800000
h100-80g     IB               64             50        6400000
b200-180g    NVLink            8            900       14400000
b200-180g    NVLink           64            900      115200000
b200-180g    IB                8            100        1600000
b200-180g    IB               64            100       12800000

读法：
  * D* 以下 = 延迟主导。此时改重叠、加 buffer 都没多少用，
    真正有效的是"合并小消息"（把 N 个小 AllReduce 合成一个）或换算法（NVLS）。
  *

[('a100-80g', 'NVLink', 8, 300.0, 4800000.0),
 ('a100-80g', 'NVLink', 64, 300.0, 38400000.0),
 ('a100-80g', 'IB', 8, 25.0, 399999.99999999994),
 ('a100-80g', 'IB', 64, 25.0, 3199999.9999999995),
 ('h100-80g', 'NVLink', 8, 450.0, 7200000.0),
 ('h100-80g', 'NVLink', 64, 450.0, 57600000.0),
 ('h100-80g', 'IB', 8, 50.0, 799999.9999999999),
 ('h100-80g', 'IB', 64, 50.0, 6399999.999999999),
 ('b200-180g', 'NVLink', 8, 900.0, 14400000.0),
 ('b200-180g', 'NVLink', 64, 900.0, 115200000.0),
 ('b200-180g', 'IB', 8, 100.0, 1599999.9999999998),
 ('b200-180g', 'IB', 64, 100.0, 12799999.999999998)]

> **两个最容易记错的结论**：
>
> 1. **IB 的 $D^*$ 只有几百 KB**（25 GB/s 档约 0.2 MB），而 NVLink 的 $D^*$ 高出两个数量级——所以「跨机通信藏不住」的第一个原因是**它本来就落在延迟主导区**，第二个原因才是带宽窄。
> 2. **DP 梯度 reduce 的消息远在 $D^*$ 之上** → 带宽主导，可重叠。而 **TP 每层每次 AR 的消息量小得多，正好落在 $D^*$ 附近** → 这就是 `tp_comm_overlap` 收益不稳定、且强依赖 shape 的原因。

## 第 2 节 · 一次迭代的通信量与重叠收益上界

**这步在做什么**：给定一套并行配置（默认 Llama-3-8B on H100，tp=8 / dp=8 / 8 卡一个节点，micro_batch=4 / grad_accum=8），算 TP 与 DP 的通信量、计算时长 $C$，然后给出四种口径下的迭代时间。

**先看一个容易被误当成结论的数字**：TP only 与 TP+SP 的通信总时长之比。这一节会算出来是 **1.00×**。

这不是模型不够精细，而是结论本身：**序列并行（SP）不改变 TP 的通信总量。** 恒等式 $\text{AR}(d) = \text{RS}(d) + \text{AG}(d)$ 决定总量守恒——baseline TP 每层 4 次 AR（每次搬 $2(n-1)/n \cdot d$），SP 每层 4 次 AG + 4 次 RS（每次搬 $(n-1)/n \cdot d$），两边都等于 $8(n-1)/n \cdot d$。

SP 分的是**激活显存**（LN / dropout / residual 从每卡存一份 $(s,b,h)$ 变成存 $(s/t,b,h)$，缩小 $t$ 倍）；真正的提速来自「显存省下来了，可以把 activation recompute 关掉」。**通信量口径上把 SP 当降本手段是常见的误解。**

In [7]:
# ---- 摘自 tools/profiling_bench/overlap_bound.py ----
def fmt_b(x):
    if x >= 1e6:
        return "%.2f MB" % (x / 1e6)
    if x >= 1e3:
        return "%.1f KB" % (x / 1e3)
    return "%.0f B" % x



def section_bound(args):
    print("=" * 78)
    print("第 2 节  一次迭代的通信量与重叠收益上界")
    print("=" * 78)

    cfg = BenchConfig(model=args.model, hardware=args.hardware,
                      tp=args.tp, cp=args.cp, pp=args.pp, dp=args.dp,
                      micro_batch=args.micro_batch, grad_accum=args.grad_accum,
                      seq_len=args.seq_len)
    hw = HARDWARE[args.hardware]
    m = MODELS[args.model]

    world = cfg.world_size
    seq = cfg.resolved_seq_len()
    params_per_card = m.n_params() / (cfg.tp * cfg.pp)
    print("配置：%s on %s，tp=%d cp=%d pp=%d dp=%d → 共 %d 卡"
          % (args.model, args.hardware, cfg.tp, cfg.cp, cfg.pp, cfg.dp, world))
    print("序列 %d，micro_batch %d，grad_accum %d"
          % (seq, cfg.micro_batch, cfg.grad_accum))
    print("每卡参数 %.0f M（bf16 即 %.0f MB 梯度）\n"
          % (params_per_card / 1e6, params_per_card * 2 / 1e6))

    # ---------------- TP 通信：直接复用 comm.py 的事件清单 ----------------
    L = m.n_layers
    print("-- TP 通信（复用 megatron_bench/comm.py 的 tensor_parallel_events）--")
    print("%-14s %-16s %10s %8s %14s %14s"
          % ("模式", "集合", "消息量", "每层次数", "每层(us)", "每迭代(ms)"))
    tp_tot = {}
    for sp in (False, True):
        if cfg.tp <= 1:
            tp_tot[sp] = 0.0
            continue
        evs = tensor_parallel_events(cfg, sequence_parallel=sp)
        tot = 0.0
        for e in evs:
            t1 = collective_time_s(e.kind, e.size_bytes, cfg.tp,
                                   hw.nvlink_gbs, args.alpha_us)
            tot += t1 * e.count_per_layer
            print("%-14s %-16s %10s %8d %14.2f %14.2f"
                  % ("TP+SP" if sp else "TP only", e.kind,
                     fmt_b(e.size_bytes), e.count_per_layer, t1 * 1e6,
                     t1 * e.count_per_layer * L * cfg.grad_accum * 1e3))
        tp_tot[sp] = tot * L * cfg.grad_accum
    if cfg.tp <= 1:
        print("%-14s %s" % ("TP only", "tp=1，无 TP 通信"))
    print("  每迭代 = 每层 × %d 层 × %d 个微批" % (L, cfg.grad_accum))
    if cfg.tp > 1:
        print("  TP only 总时长 %.2f ms ；TP+SP 总时长 %.2f ms ；比值 %.2f×"
              % (tp_tot[False] * 1e3, tp_tot[True] * 1e3,
                 tp_tot[False] / max(tp_tot[True], 1e-12)))
        print("""
  → 比值恒为 1.00×，这不是模型不够精细，而是结论本身：
    **序列并行（SP）不改变 TP 的通信总量。**
    恒等式 AR(d) = RS(d) + AG(d)：baseline TP 每层 4 次 AR
    （每次搬 2(n−1)/n·d），SP 每层 4 次 AG + 4 次 RS（每次搬 (n−1)/n·d），
    两边都等于 8(n−1)/n·d。SP 分的是**激活显存**——LN / dropout / residual
    从每卡存一份 (s,b,h) 变成存 (s/t,b,h)，缩小 t 倍；
    真正的提速来自"显存省下来了，可以把 activation recompute 关掉"。
    通信量口径上把 SP 当降本手段是常见的误解。
    见 Korthikanti et al. arXiv:2205.05198；通信条数的独立复述见
    arXiv:2311.02382 §3（"8 global communications per attention layer：
    4 in forward pass and 4 in backward pass"）。
    本条已用于校准 megatron_bench/comm.py 的 tensor_parallel_events()。
""")

    # ---------------- DP 梯度 reduce ----------------
    h, ffn_onepart = m.hidden, m.ffn_hidden
    attn_bytes = (h * h + 2 * h * m.kv_dim + h * h) * 2 / cfg.tp
    mlp_bytes = ((2 * h * ffn_onepart if m.gated_mlp else h * ffn_onepart)
                 + ffn_onepart * h) * 2 / cfg.tp
    per_layer_bytes = attn_bytes + mlp_bytes

    dp_kind = "reduce_scatter" if cfg.use_distributed_optimizer else "all_reduce"
    dp_link = hw.ib_gbs if world > hw.gpus_per_node else hw.nvlink_gbs
    dp_per_layer = collective_time_s(dp_kind, per_layer_bytes, cfg.dp,
                                    dp_link, args.alpha_us)
    dp_total = dp_per_layer * L
    print("-- DP 梯度 reduce（%s，n=%d，链路 %.0f GB/s）--"
          % (dp_kind, cfg.dp, dp_link))
    print("  每层参数字节 %s（已按 tp=%d 分片，pp 不计入 DP 组）"
          % (fmt_b(per_layer_bytes), cfg.tp))
    print("  每层耗时 %.3f ms × %d 层 = 每迭代 %.3f ms"
          % (dp_per_layer * 1e3, L, dp_total * 1e3))
    # 对账：朴素估算 = 本卡梯度字节 × (dp-1)/dp / 链路带宽
    naive_bytes = per_layer_bytes * L * (cfg.dp - 1) / cfg.dp
    naive_t = naive_bytes / (dp_link * 1e9)
    print("  对账（朴素估算，只算带宽项）：%s × (dp−1)/dp = %s，%.3f ms"
          % (fmt_b(per_layer_bytes * L), fmt_b(naive_bytes), naive_t * 1e3))
    print("  差额 %.3f ms 全部是 %d 层 × %d 步的 α 项；相对偏差 %.2f%%"
          % ((dp_total - naive_t) * 1e3, L, cfg.dp - 1,
             (dp_total - naive_t) / naive_t * 100.0))

    # ---------------- 计算量 ----------------
    tokens = cfg.micro_batch * seq * cfg.grad_accum * cfg.dp
    flops = 6.0 * m.n_params() * tokens
    C = flops / (world * hw.bf16_tflops * 1e12 * args.mfu)
    print("\n-- 计算量 --")
    print("  每迭代 token 数 = micro_batch %d × seq %d × grad_accum %d × dp %d = %s"
          % (cfg.micro_batch, seq, cfg.grad_accum, cfg.dp,
             "{:,}".format(int(tokens))))
    print("  6ND FLOPs = %.3e" % flops)
    print("  计算时长 C = FLOPs / (%d 卡 × %.0f TFLOP/s × MFU %.0f%%) = %.3f s"
          % (world, hw.bf16_tflops, args.mfu * 100, C))

    # ---------------- 重叠收益上界 ----------------
    M = tp_tot.get(False, 0.0) + dp_total
    print("\n-- 重叠收益上界（M = TP %.3f + DP %.3f = %.3f s，C = %.3f s）--"
          % (tp_tot.get(False, 0.0), dp_total, M, C))
    base = C + M
    print("%-38s %14s %12s" % ("口径", "迭代时间(s)", "相对不重叠"))
    print("%-38s %14.4f %11.1f%%" % ("完全不重叠  T = C + M", base, 100.0))
    for eta in (0.3, 0.5, 0.7, 0.9, 1.0):
        t = C + M - eta * min(C, M)
        print("%-38s %14.4f %11.1f%%"
              % ("重叠效率 η=%.1f  T = C+M−η·min(C,M)" % eta, t,
                 t / base * 100.0))
    floor = max(C, M)
    print("%-38s %14.4f %11.1f%%"
          % ("完美重叠  T = max(C, M)", floor, floor / base * 100.0))
    print("""读法：
  * `max(C, M)` 是任何调度器都突破不了的下界——**它是上界收益，不是目标值**。
  * η 从 0 到 1 的落差 = 「重叠这件事的全部价值」= %.1f%%。若只有几个百分点，
    所有 overlap 调参都不值得做，瓶颈在别处（通常是 C 本身或 idle）。
  * 本例 min(C, M) = M，说明通信总量小于计算总量：**重叠在原理上够用**，
    那么"通信没藏住"就一定是调度/依赖问题，而不是通信量太大。
  * 反过来若 M > C，即使 η=1 也只能降到 M：此时唯一有效的手段是减少通信量
    本身（换并行策略、增大 micro batch 摊薄、减少重计算触发的额外通信、
    改通信算法/拓扑）。注意**开序列并行不在此列**——它不改变通信总量（见上）。
  * 实测 η 的算法：从 trace 里量 `overlap_time / min(C, M)`，
    即 `trace_agg.py` 报的 `overlap / comm_total`（在 C ≥ M 时等价）。"""
          % ((base - floor) / base * 100.0))
    dump_json = {
        "config": {"model": args.model, "hardware": args.hardware,
                   "tp": cfg.tp, "pp": cfg.pp, "dp": cfg.dp,
                   "world_size": world, "mfu": args.mfu},
        "M_tp_s": tp_tot.get(False, 0.0), "M_dp_s": dp_total, "M_s": M,
        "C_s": C, "no_overlap_s": base, "perfect_overlap_floor_s": floor,
        "overlap_value_pct": (base - floor) / base * 100.0,
    }
    if args.dump:
        import json
        with open(args.dump, "w", encoding="utf-8") as f:
            json.dump(dump_json, f, indent=2, ensure_ascii=False)
        print("\n结果已写出: %s" % args.dump)



section_bound(ARGS)

第 2 节  一次迭代的通信量与重叠收益上界
配置：llama3-8b on h100-80g，tp=8 cp=1 pp=1 dp=8 → 共 64 卡
序列 8192，micro_batch 4，grad_accum 8
每卡参数 1004 M（bf16 即 2007 MB 梯度）

-- TP 通信（复用 megatron_bench/comm.py 的 tensor_parallel_events）--
模式             集合                      消息量     每层次数         每层(us)        每迭代(ms)
TP only        all_reduce        268.44 MB        2        1071.92         548.82
TP only        all_reduce        268.44 MB        2        1071.92         548.82
TP+SP          all_gather        268.44 MB        4         535.96         548.82
TP+SP          reduce_scatter    268.44 MB        4         535.96         548.82
  每迭代 = 每层 × 32 层 × 8 个微批
  TP only 总时长 1097.64 ms ；TP+SP 总时长 1097.64 ms ；比值 1.00×

  → 比值恒为 1.00×，这不是模型不够精细，而是结论本身：
    **序列并行（SP）不改变 TP 的通信总量。**
    恒等式 AR(d) = RS(d) + AG(d)：baseline TP 每层 4 次 AR
    （每次搬 2(n−1)/n·d），SP 每层 4 次 AG + 4 次 RS（每次搬 (n−1)/n·d），
    两边都等于 8(n−1)/n·d。SP 分的是**激活显存**——LN / dropout / residual
    从每卡存一份 (s,b,h) 变成存 (s/t,b,h)，缩小 t 倍；
    真正的提速来自"显存省下来了，可以把 

> **怎么读这张上界表**：
>
> - `max(C, M)` 是任何调度器都突破不了的下界——**它是上界收益，不是目标值**。
> - $\eta$ 从 0 到 1 的落差 = 「重叠这件事的全部价值」。若只有几个百分点，所有 overlap 调参都不值得做，瓶颈在别处。
> - 本例 $\min(C, M) = M$，说明通信总量小于计算总量：**重叠在原理上够用**，那么「通信没藏住」就一定是调度/依赖问题，而不是通信量太大。
> - 反过来若 $M > C$，即使 $\eta = 1$ 也只能降到 $M$：此时唯一有效的手段是**减少通信量本身**（换并行策略、增大 micro batch 摊薄、改通信算法/拓扑）。注意**开序列并行不在此列**。
> - 实测 $\eta$ 的算法：从 trace 里量 `overlap_time / min(C, M)`。

## 第 3 节 · nccl-tests：algbw 与 busbw 差多少、能反推什么

**这步在做什么**：$\text{busbw} = \text{algbw} \times \text{factor}$，factor 只取决于集合类型与卡数。给出换算表，并演示**algbw 的排名与 busbw 的排名可以相反**。

In [8]:
# ---- 摘自 tools/profiling_bench/overlap_bound.py ----
def busbw_factor(kind, n):
    """nccl-tests 定义：busbw = algbw × factor。"""
    if n <= 1:
        return 1.0
    if kind in ("all_reduce",):
        return 2.0 * (n - 1) / n
    if kind in ("all_gather", "reduce_scatter"):
        return (n - 1) / n
    if kind in ("all_to_all", "reduce", "broadcast"):
        return (n - 1) / n
    raise ValueError(kind)



def section_busbw():
    print("\n" + "=" * 78)
    print("第 3 节  nccl-tests：algbw 与 busbw 差多少、能反推什么")
    print("=" * 78)
    print("busbw = algbw × factor，factor 只取决于集合类型与卡数：\n")
    print("%-18s %10s %10s %12s %12s" %
          ("集合", "N", "factor", "algbw", "busbw"))
    for kind in ("all_reduce", "all_gather", "reduce_scatter"):
        for n in (2, 4, 8, 16, 64):
            f = busbw_factor(kind, n)
            algbw = 10.0  # 假想读数，用于展示换算
            print("%-18s %10d %10.3f %12.2f %12.2f"
                  % (kind, n, f, algbw, algbw * f))
    print("""
读法：
  * 同一份数据，AllReduce 的 factor（2(N−1)/N）是 AllGather 的两倍。
    所以"我的 AllReduce 达到 40 GB/s"和"我的 AllGather 达到 40 GB/s"
    在物理链路上完全不是一回事。
  * N=2 时 AllReduce 的 factor 只有 1.0 —— **两卡 AllReduce 就是 SendRecv，
    没有 busbw 口径优势**。拿两卡测出来的 busbw 去推 8 卡性能会严重高估。
  * **busbw 是双向口径，链路标称带宽通常是单向口径**。所以 ring AllReduce 在
    大 N 下 busbw ≈ 2 × 单向链路是**正常的**（N=8 时 1.75×），
    看到它超过链路标称值不要急着报故障。
  * 真正会让口径失效的是 NVLS（NVLink SHARP，网内归约）、CollNet（SHARP
    交换机内归约）与分层算法：数据不在链路上被逐跳转发，甚至不在链路上归约，
    此时上面这个 factor 的推导前提（数据在 N 卡链路上都跑了一遍）不成立，
    busbw 会超出 2× 这个经验上界。**判据：busbw / 单向链路 > 2 时，
    不要再拿它反推线速，只能同类算法、同 N 之间横向比。**
  * 正确用法：同算法、同 N、同消息量下横比。busbw 曲线在某档位出现 cliff，
    就把故障定位到了"引入该档位的那一层"（小消息看 α、中消息看算法、
    大消息看链路）。""")

    # 演示：algbw 排名与 busbw 排名可以相反
    print("\n-- 演示：algbw 的排名与 busbw 的排名可以相反 --")
    link = 25.0
    print("%-16s %6s %10s %10s %12s %12s" %
          ("集合", "N", "algbw", "factor", "busbw", "busbw/单向链路"))
    cases = (("all_reduce", 8, 22.0), ("all_gather", 8, 39.0),
             ("all_reduce", 2, 24.0))
    for kind, n, algbw in cases:
        f = busbw_factor(kind, n)
        b = algbw * f
        print("%-16s %6d %10.1f %10.3f %12.1f %13.2fx"
              % (kind, n, algbw, f, b, b / link))
    print("""   看 algbw：AllGather 39.0 > AllReduce 22.0 → "AllGather 快得多"。
   看 busbw：AllReduce 38.5 > AllGather 34.1 → "AllReduce 快得多"。
   两个结论都"对"，也都没用——它们量的不是同一个东西。
   AllReduce 的 algbw 低是因为它按定义要搬 2(N−1)/N 倍的量；
   AllGather 的 factor < 1 是因为 algbw 的分母是"集合的数据量"，
   而 ring AllGather 的分母口径下每卡实际只承担 (N−1)/N 份。
   **跨集合类型比带宽数字，是这张表要防的唯一一件事。**""")
    return



section_busbw()


第 3 节  nccl-tests：algbw 与 busbw 差多少、能反推什么
busbw = algbw × factor，factor 只取决于集合类型与卡数：

集合                          N     factor        algbw        busbw
all_reduce                  2      1.000        10.00        10.00
all_reduce                  4      1.500        10.00        15.00
all_reduce                  8      1.750        10.00        17.50
all_reduce                 16      1.875        10.00        18.75
all_reduce                 64      1.969        10.00        19.69
all_gather                  2      0.500        10.00         5.00
all_gather                  4      0.750        10.00         7.50
all_gather                  8      0.875        10.00         8.75
all_gather                 16      0.938        10.00         9.38
all_gather                 64      0.984        10.00         9.84
reduce_scatter              2      0.500        10.00         5.00
reduce_scatter              4      0.750        10.00         7.50
reduce_scatter              8      0.875  

> **口径陷阱**：
>
> - **N=2 时 AllReduce 的 factor 只有 1.0**——两卡 AllReduce 就是 SendRecv，没有 busbw 口径优势。拿两卡测出来的 busbw 去推 8 卡性能会严重高估。
> - **busbw 是双向口径，链路标称带宽通常是单向口径**。所以 ring AllReduce 在大 N 下 busbw $\approx 2 \times$ 单向链路是**正常的**（N=8 时 1.75×），看到它超过链路标称值不要急着报故障。
> - 真正会让口径失效的是 **NVLS / CollNet / 分层算法**：此时上面这个 factor 的推导前提（数据在 N 卡链路上都跑了一遍）不成立，busbw 会超出 2× 这个经验上界。**判据：busbw / 单向链路 > 2 时，不要再拿它反推线速。**
> - **跨集合类型比带宽数字，是这张表要防的唯一一件事。**

## 参考文献

- [nccl-tests 源码](https://github.com/NVIDIA/nccl-tests)
- Korthikanti et al., *Reducing Activation Recomputation in Large Transformer Models*, [arXiv:2205.05198](https://arxiv.org/abs/2205.05198) —— 序列并行
- 通信条数的独立复述见 [arXiv:2311.02382](https://arxiv.org/abs/2311.02382) §3